# Hadron AQC-Tensor

Compress the SU(2) hadron circuit (IBM tutorial) with AQC-Tensor: build the target state as an MPS, then train a much shorter circuit to reproduce it.

**12 qubits**
1. Check our MPS code
2. Build the target
3. Hand the target to IBM's library
4. Build the cheap trainable circuit
5. Train the AQC angles to match the TEBD target
6. Count CNOTs and depth
7. Target with a bond cap, and a health check
8. Calibrate the convergence check

**32 qubits** *(planned)*

**80 qubits** *(planned, optional)*

**120 qubits**
1. Build the targets (switch: `build_120_qubit_targets`)

Each size is its own section: click the ⌄ arrow left of a `#` heading to fold it. Each section ends with a short summary. The trainable circuit built by hand (what the library does inside) is in `hadron-aqc-mapping.ipynb`.

## Setup
Imports and shared functions (`hadron_aqc_utils.py`).

In [1]:
%load_ext autoreload
%autoreload 2

import numpy as np
from qiskit.quantum_info import Statevector

from hadron_aqc_utils import (
    # hadron circuit
    construct_circuit,
    # our own MPS code (TEBD)
    mps_to_statevector, mps_overlap, circuit_to_mps_tensors,
    # settings shared by every size
    KINETIC_STRENGTH, ELECTRIC_FIELD_STRENGTH, MASS,
    # library route: the AQC recipe, written once (IBM qiskit-addon-aqc-tensor + quimb)
    make_simulator_settings, tensornetwork_from_circuit, mps_fidelity, tensors_to_library_target,
    build_target_using_lib, check_target, build_cheap_trainable_circuit, train_to_target,
    count_cnots, count_and_verify,
)

## Switches
Long runs and file writes are off by default, so "Run All" is always safe (about 2 minutes). Set a switch to `True` only when you mean it.

In [25]:
# Switches — anything long, or anything that writes files, is OFF unless set to True here
# 12 qubits
save_12_qubit_trained_angles = False   # Step 5: overwrite trained_aqc_hadron_12q_k3.npz (training itself always runs, ~20 s)
# 32 qubits
check_32_qubit_convergence = False   # Step 1: full convergence check, max_bond 64-512 (~6 min). Off = build only the training target (~1 min)
train_32_qubits = False
# 80 qubits (planned, meson only)
build_80_qubit_target = False
train_80_qubits = False
# 120 qubits
build_120_qubit_targets = False        # Step 1: rebuild the targets into targets_120qubits_merged/ (about 6.5 hours)


# 12 qubits (6 sites, 10 Trotter steps)
Compress the circuit, and calibrate the convergence check where the exact answer is known.

## Settings (12 qubits)
Everything this section changes, in one place.

In [3]:
# Settings (12 qubits)
n_sites = 6
num_qubits = 2 * n_sites
num_trotter_steps = 10      # the target circuit
num_coarse_steps = 3        # the cheap circuit that gets trained
kinetic_strength, electric_field_strength, mass = KINETIC_STRENGTH, ELECTRIC_FIELD_STRENGTH, MASS

## Step 1 — Check our MPS code (12 qubits)
Run the 10-step circuit with our own MPS code and compare with the exact statevector.

In [4]:
# Step 1 — check our MPS code against the exact statevector
# compare meson and vaccum circuits in one shot

for use_meson_not_vacuum in [False, True]:
    qc = construct_circuit(n_sites, num_trotter_steps, kinetic_strength, electric_field_strength, mass, use_meson_not_vacuum=use_meson_not_vacuum)
    zero_state_mps = [np.array([1,0], dtype=complex).reshape(1,2,1) for _ in range(num_qubits)] # create all |0> and make it tensors for 12 qubits case
    mps_tensors_after_circuit_evolve = circuit_to_mps_tensors(zero_state_mps, qc)

    state_from_mps = mps_to_statevector(mps_tensors_after_circuit_evolve)
    state_from_exact = Statevector(qc).data

    max_difference = np.max(np.abs(state_from_mps - state_from_exact))
    fidelity = abs(np.vdot(state_from_exact, state_from_mps)) ** 2
    label = "meson" if use_meson_not_vacuum else "vacuum"
    print(f"{label:6s}: max difference = {max_difference:.1e}, fidelity = {fidelity:.12f}")



vacuum: max difference = 3.6e-15, fidelity = 1.000000000000
meson : max difference = 4.3e-15, fidelity = 1.000000000000


## Step 2 — Build the target (12 qubits)
Our own TEBD runs the 10-step circuit, for vacuum and meson. Nothing is cut at 12 qubits.

In [5]:
# Step 2 — build the 10-step targets (vacuum and meson)
def build_target_tensors(use_meson_not_vacuum):
    qc = construct_circuit(n_sites, num_trotter_steps, kinetic_strength,
                           electric_field_strength, mass,
                           use_meson_not_vacuum=use_meson_not_vacuum)
    zero_state_mps = [np.array([1, 0], dtype=complex).reshape(1, 2, 1) for _ in range(num_qubits)]
    return circuit_to_mps_tensors(zero_state_mps, qc)

vacuum_target_tensors = build_target_tensors(use_meson_not_vacuum=False)
meson_target_tensors = build_target_tensors(use_meson_not_vacuum=True)

print(f"vacuum norm = {abs(mps_overlap(vacuum_target_tensors, vacuum_target_tensors)):.12f}")
print(f"meson  norm = {abs(mps_overlap(meson_target_tensors, meson_target_tensors)):.12f}")
print(f"vacuum vs meson overlap = {abs(mps_overlap(vacuum_target_tensors, meson_target_tensors))**2:.1e}")


vacuum norm = 1.000000000000
meson  norm = 1.000000000000
vacuum vs meson overlap = 7.7e-30


## Step 3 — Hand the target to IBM's library (12 qubits)
Repackage our tensors in the library's format, and check they match the library's own TEBD.

In [6]:
# Step 3 — hand our targets to IBM's library, and check them against the library's own TEBD
simulator_settings = make_simulator_settings()   # no bond cap at 12 qubits

vacuum_target_mps = tensors_to_library_target(vacuum_target_tensors)
meson_target_mps = tensors_to_library_target(meson_target_tensors)

for use_meson_not_vacuum in [False, True]:
    label = "meson" if use_meson_not_vacuum else "vacuum"
    our_target = meson_target_mps if use_meson_not_vacuum else vacuum_target_mps
    library_target = build_target_using_lib(n_sites=n_sites, num_trotter_steps=num_trotter_steps, max_bond=None, use_meson_not_vacuum=use_meson_not_vacuum)
    print(f"{label:6s}: our TEBD target vs library's own TEBD target = {mps_fidelity(our_target, library_target):.12f}")

vacuum: our TEBD target vs library's own TEBD target = 1.000000000000
meson : our TEBD target vs library's own TEBD target = 1.000000000000


## Step 4 — Build the cheap trainable circuit (12 qubits)
A 3-step version of the same circuit. The library turns it into a trainable circuit whose starting angles copy it exactly.

In [7]:
# Step 4 — cheap 3-step circuit -> the library builds the trainable circuit + its starting angles
aqc_ansatz = {}
aqc_starting_angles = {}
for use_meson_not_vacuum in [False, True]:
    label = "meson" if use_meson_not_vacuum else "vacuum"
    aqc_ansatz[label], aqc_starting_angles[label] = build_cheap_trainable_circuit(n_sites, num_trotter_steps, num_coarse_steps, use_meson_not_vacuum)

    untrained_mps = tensornetwork_from_circuit(aqc_ansatz[label].assign_parameters(aqc_starting_angles[label]), simulator_settings)
    target_mps = meson_target_mps if use_meson_not_vacuum else vacuum_target_mps
    print(f"{label:6s}: {len(aqc_starting_angles[label])} angles, before-training fidelity vs target = {mps_fidelity(untrained_mps, target_mps):.6f}")

vacuum: 699 angles, before-training fidelity vs target = 0.851366
meson : 699 angles, before-training fidelity vs target = 0.899267


## Step 5 — Train the AQC angles to match the TEBD target (12 qubits)
Turn the angles until the trainable circuit matches the target.

In [8]:
# Step 5 — train
trained_angles = {}
for use_meson_not_vacuum in [False, True]:
    label = "meson" if use_meson_not_vacuum else "vacuum"
    target_mps = meson_target_mps if use_meson_not_vacuum else vacuum_target_mps

    result = train_to_target(aqc_ansatz[label], aqc_starting_angles[label], target_mps, simulator_settings)
    trained_angles[label] = result.x

    trained_mps = tensornetwork_from_circuit(aqc_ansatz[label].assign_parameters(result.x), simulator_settings)
    print(f"{label:6s}: after-training fidelity vs target = {mps_fidelity(trained_mps, target_mps):.6f}  ({result.nit} training steps)")

if save_12_qubit_trained_angles:
    np.savez("trained_aqc_hadron_12q_k3.npz", vacuum=trained_angles["vacuum"], meson=trained_angles["meson"])
    print("saved trained_aqc_hadron_12q_k3.npz")

vacuum: after-training fidelity vs target = 0.999967  (14 training steps)
meson : after-training fidelity vs target = 0.999986  (25 training steps)


## Step 6 — Count CNOTs and depth (12 qubits)
Both circuits transpiled the same way: linear chain, optimisation level 3.

In [9]:
# Step 6 — CNOTs and depth (linear chain, optimisation level 3), and the fidelity of the TRANSPILED circuit
deep_circuit = construct_circuit(n_sites, num_trotter_steps, kinetic_strength, electric_field_strength, mass)
cnots, two_qubit_depth, total_depth, _ = count_cnots(deep_circuit)
print(f"10-step circuit (the target): {cnots} CNOTs, two-qubit depth {two_qubit_depth}, total depth {total_depth}")

for label in ["vacuum", "meson"]:
    target_mps = meson_target_mps if label == "meson" else vacuum_target_mps
    counts = count_and_verify(aqc_ansatz[label], trained_angles[label], target_mps, simulator_settings)
    print(f"trained AQC ({label}): {counts['cnots']} CNOTs, two-qubit depth {counts['two_qubit_depth']}, total depth {counts['total_depth']}")
    print(f"   qubit order unchanged by transpiling: {counts['qubit_order_unchanged']},  fidelity of the TRANSPILED circuit vs target = {counts['transpiled_fidelity']:.6f}")

10-step circuit (the target): 611 CNOTs, two-qubit depth 147, total depth 347
trained AQC (vacuum): 201 CNOTs, two-qubit depth 47, total depth 165
   qubit order unchanged by transpiling: True,  fidelity of the TRANSPILED circuit vs target = 0.999967
trained AQC (meson): 202 CNOTs, two-qubit depth 48, total depth 163
   qubit order unchanged by transpiling: True,  fidelity of the TRANSPILED circuit vs target = 0.999986


## Step 7 — Target with a bond cap, and a health check (12 qubits)
`build_target_using_lib`: merge the gates on each qubit pair into one block, then quimb builds the MPS, cutting each link to `max_bond`.
`check_target`: norm violation, chance each qubit reads 1, count of filled seats (must equal `n_sites`).

In [10]:
# Step 7 — target with a bond cap, and a health check
# build_target_using_lib and check_target live in hadron_aqc_utils.py
for max_bond in [None, 16]:
    target_mps = build_target_using_lib(n_sites=n_sites, num_trotter_steps=num_trotter_steps, max_bond=max_bond, use_meson_not_vacuum=False)
    norm_violation, chance_of_filled_per_seat, count_of_filled_seats = check_target(target_mps)
    print(f"max_bond={max_bond}: norm violation = {norm_violation:.4f}, count of filled seats = {count_of_filled_seats:.4f} (must be {n_sites})")
    print("   chance each seat is filled:", " ".join(f"{c:.3f}" for c in chance_of_filled_per_seat))

max_bond=None: norm violation = -0.0000, count of filled seats = 6.0000 (must be 6)
   chance each seat is filled: 0.613 0.613 0.454 0.454 0.313 0.313 0.687 0.687 0.546 0.546 0.387 0.387
max_bond=16: norm violation = 0.0165, count of filled seats = 6.0000 (must be 6)
   chance each seat is filled: 0.613 0.613 0.457 0.457 0.282 0.282 0.718 0.718 0.543 0.543 0.387 0.387


## Step 8 — Calibrate the convergence check (12 qubits)
Double `max_bond` and compare with the exact answer. The change when `max_bond` doubles shows the error of the smaller one.

In [11]:
# Step 8 — calibrate the convergence check
#The middle link has the biggest smaller side (half the qubits), so:
#largest possible max_bond = 2^(number of qubits / 2)

max_bonds = [8, 16, 32, 64]

for use_meson_not_vacuum in [False, True]:
    label = "meson" if use_meson_not_vacuum else "vacuum"
    exact_target = build_target_using_lib(n_sites=n_sites, num_trotter_steps=num_trotter_steps, max_bond=None, use_meson_not_vacuum=use_meson_not_vacuum)
    _, exact_chances_filled_per_seat, _ = check_target(exact_target)

    previous_chances_filled_per_seat = None

    for max_bond in max_bonds:
        per_bond_mps = build_target_using_lib(n_sites=n_sites, num_trotter_steps=num_trotter_steps, max_bond=max_bond, use_meson_not_vacuum=use_meson_not_vacuum)
        norm_violation, bond_chances_filled_per_seat, count_of_filled_seats = check_target(per_bond_mps)
        fidelity_vs_exact = mps_fidelity(per_bond_mps, exact_target) / (1 - norm_violation)

        error_vs_exact_chances = max(abs(np.array(bond_chances_filled_per_seat) - np.array(exact_chances_filled_per_seat)))

        change_vs_previous_chances = "  -   " if previous_chances_filled_per_seat is None else f"{max(abs(np.array(bond_chances_filled_per_seat) - np.array(previous_chances_filled_per_seat))):.4f}"
        
        print(f"{label:6s} max_bond={max_bond:2d}: fidelity={fidelity_vs_exact:.4f}  norm violation={norm_violation:.4f}  count of filled seats={count_of_filled_seats:.4f}  "f"biggest seat error vs exact={error_vs_exact_chances:.4f}  biggest seat change vs previous cap={change_vs_previous_chances}")
        previous_chances_filled_per_seat = bond_chances_filled_per_seat

vacuum max_bond= 8: fidelity=0.2094  norm violation=0.2914  count of filled seats=6.0000  biggest seat error vs exact=0.3857  biggest seat change vs previous cap=  -   
vacuum max_bond=16: fidelity=0.9406  norm violation=0.0165  count of filled seats=6.0000  biggest seat error vs exact=0.0309  biggest seat change vs previous cap=0.4166
vacuum max_bond=32: fidelity=0.9855  norm violation=0.0055  count of filled seats=6.0000  biggest seat error vs exact=0.0043  biggest seat change vs previous cap=0.0269
vacuum max_bond=64: fidelity=1.0000  norm violation=-0.0000  count of filled seats=6.0000  biggest seat error vs exact=0.0000  biggest seat change vs previous cap=0.0043
meson  max_bond= 8: fidelity=0.8734  norm violation=0.0397  count of filled seats=6.0000  biggest seat error vs exact=0.0296  biggest seat change vs previous cap=  -   
meson  max_bond=16: fidelity=1.0000  norm violation=0.0000  count of filled seats=6.0000  biggest seat error vs exact=0.0000  biggest seat change vs previ

## Summary (12 qubits)
| | vacuum | meson |
|---|---|---|
| fidelity after training (same after transpiling) | 0.999967 | 0.999986 |
| CNOTs: 10-step → trained | 611 → 201 | 611 → 202 |
| two-qubit depth | 147 → 47 | 147 → 48 |
| total depth | 347 → 165 | 347 → 163 |

# 32 qubits (16 sites, 10 Trotter steps)
The size where compression failed before.

1. Build the target
2. Check convergence
3. Build the cheap trainable circuit
4. Train the AQC angles to match the target
5. Count CNOTs and depth

## Settings (32 qubits)
Everything this section changes, in one place.

In [14]:
# Settings (32 qubits)
n_sites_32q = 16
num_qubits_32q = 2 * n_sites_32q
num_trotter_steps_32q = 10            # the target circuit
num_coarse_steps_32q = 3              # the cheap circuit that gets trained
max_bonds_32q = [64, 128, 256, 512]   # step 1: convergence check
target_max_bond_32q = 256             # the target used for training
training_max_bond_32q = 128           # cap on the trainable circuit while training

## Step 1 — Build the targets and check convergence (32 qubits)
Build vacuum and meson at doubling `max_bond`; the change when it doubles shows the error of the smaller one. The target at `target_max_bond_32q` is kept for training.

In [17]:
# Step 1 (32 qubits) — build the targets and check convergence
import time

targets_32q = {}
bonds_to_run = max_bonds_32q if check_32_qubit_convergence else [target_max_bond_32q]

for use_meson_not_vacuum in [False, True]:
    label = "meson" if use_meson_not_vacuum else "vacuum"
    previous_chances_filled_per_seat = None

    for max_bond in bonds_to_run:
        start_time = time.time()
        target_mps = build_target_using_lib(n_sites=n_sites_32q, num_trotter_steps=num_trotter_steps_32q, max_bond=max_bond, use_meson_not_vacuum=use_meson_not_vacuum)
        norm_violation, chances_filled_per_seat, count_of_filled_seats = check_target(target_mps)

        change_vs_previous_chances = "  -   " if previous_chances_filled_per_seat is None else f"{max(abs(np.array(chances_filled_per_seat) - np.array(previous_chances_filled_per_seat))):.4f}"
        print(f"{label:6s} max_bond={max_bond:3d}: {time.time() - start_time:4.0f} s  norm violation={norm_violation:.4f}  "
              f"count of filled seats={count_of_filled_seats:.4f} (must be {n_sites_32q})  biggest seat change vs previous cap={change_vs_previous_chances}")
        previous_chances_filled_per_seat = chances_filled_per_seat

        if max_bond == target_max_bond_32q: #These two lines keep the one target we'll train on, and throw away the others.
            targets_32q[label] = target_mps

vacuum max_bond= 64:    3 s  norm violation=0.0021  count of filled seats=16.0000 (must be 16)  biggest seat change vs previous cap=  -   
vacuum max_bond=128:    9 s  norm violation=0.0000  count of filled seats=16.0000 (must be 16)  biggest seat change vs previous cap=0.0001
vacuum max_bond=256:   33 s  norm violation=0.0000  count of filled seats=16.0000 (must be 16)  biggest seat change vs previous cap=0.0000
vacuum max_bond=512:  141 s  norm violation=0.0000  count of filled seats=16.0000 (must be 16)  biggest seat change vs previous cap=0.0000
meson  max_bond= 64:    3 s  norm violation=0.0006  count of filled seats=16.0000 (must be 16)  biggest seat change vs previous cap=  -   
meson  max_bond=128:   10 s  norm violation=0.0000  count of filled seats=16.0000 (must be 16)  biggest seat change vs previous cap=0.0001
meson  max_bond=256:   31 s  norm violation=0.0000  count of filled seats=16.0000 (must be 16)  biggest seat change vs previous cap=0.0000
meson  max_bond=512:  139 s

## Step 2 — Build the cheap trainable circuit (32 qubits)
The 3-step circuit (strengths × 10/3); the library turns it into a trainable circuit whose starting angles copy it exactly. Simulated with the bond cap `training_max_bond_32q`.

In [20]:
# Step 2 (32 qubits) — cheap 3-step circuit -> the library builds the trainable circuit + its starting angles
training_settings_32q = make_simulator_settings(max_bond=training_max_bond_32q)

aqc_ansatz_32q = {}
aqc_starting_angles_32q = {}
for use_meson_not_vacuum in [False, True]:
    label="meson" if use_meson_not_vacuum else "vacuum"
    aqc_ansatz_32q[label], aqc_starting_angles_32q[label] = build_cheap_trainable_circuit(n_sites_32q, num_trotter_steps_32q, num_coarse_steps_32q, use_meson_not_vacuum)

    untrained_mps = tensornetwork_from_circuit(aqc_ansatz_32q[label].assign_parameters(aqc_starting_angles_32q[label]), training_settings_32q)

    print(f"{label:6s}: {len(aqc_starting_angles_32q[label])} angles, before-training fidelity vs target = {mps_fidelity(untrained_mps, targets_32q[label]):.6f}  (target max_bond={target_max_bond_32q})")


vacuum: 2044 angles, before-training fidelity vs target = 0.642781  (target max_bond=256)
meson : 2044 angles, before-training fidelity vs target = 0.671261  (target max_bond=256)


## Step 3 — Train the AQC angles to match the target (32 qubits)
Trains with the trainable circuit capped at `training_max_bond_32q` and saves the angles (switch: `train_32_qubits`). With the switch off, loads the saved angles instead.

In [23]:
# Step 3 (32 qubits) — train (switch: train_32_qubits); with the switch off, load the saved angles
import os

trained_angles_file_32q = "trained_aqc_hadron_32q_k3.npz"
trained_angles_32q = {}

if train_32_qubits:
    for use_meson_not_vacuum in [False, True]:
        label = "meson" if use_meson_not_vacuum else "vacuum"
        start_time = time.time()
        result = train_to_target(aqc_ansatz_32q[label], aqc_starting_angles_32q[label], targets_32q[label], training_settings_32q)
        trained_angles_32q[label] = result.x

        trained_mps = tensornetwork_from_circuit(aqc_ansatz_32q[label].assign_parameters(result.x), training_settings_32q)
        print(f"{label:6s}: after-training fidelity vs target = {mps_fidelity(trained_mps, targets_32q[label]):.6f}  "
              f"({result.nit} training steps, {(time.time() - start_time) / 60:.1f} min)")
    np.savez(trained_angles_file_32q, **trained_angles_32q)
    print("saved", trained_angles_file_32q)
elif os.path.exists(trained_angles_file_32q):
    saved = np.load(trained_angles_file_32q)
    trained_angles_32q = {label: saved[label] for label in ["vacuum", "meson"]}
    print("loaded", trained_angles_file_32q, "(set train_32_qubits = True to retrain)")
else:
    print("Step 3 skipped: set train_32_qubits = True in the Switches cell to train (about 10 min per state).")


vacuum: after-training fidelity vs target = 0.999000  (93 training steps, 4.1 min)
meson : after-training fidelity vs target = 0.999499  (95 training steps, 3.8 min)
saved trained_aqc_hadron_32q_k3.npz


## Step 4 — Count CNOTs and depth, and verify (32 qubits)
CNOTs and depth after transpiling, the fidelity of the transpiled circuit, and the trained circuit rechecked at bond 256 / 512 (the training used a cap of 128).

In [30]:
# Step 4 (32 qubits) — CNOTs and depth, fidelity of the transpiled circuit, and a recheck at higher bond
for label in ["vacuum", "meson"]:
    circuit_32q = construct_circuit(n_sites_32q, num_trotter_steps_32q, kinetic_strength, electric_field_strength, mass, use_meson_not_vacuum=(label == "meson"))
    cnots, two_qubit_depth, total_depth, _ = count_cnots(circuit_32q)
    print(f"\n{num_qubits_32q} qubits, {label}, original {num_trotter_steps_32q}-step target circuit: {cnots} CNOTs, two-qubit depth {two_qubit_depth}, total depth {total_depth}"
          f"  (transpiled to a linear chain, optimisation level 3)")

    counts = count_and_verify(aqc_ansatz_32q[label], trained_angles_32q[label], targets_32q[label], training_settings_32q)
    print(f"{num_qubits_32q} qubits, {label}, trained AQC circuit ({num_coarse_steps_32q}-step start): {counts['cnots']} CNOTs, "
          f"two-qubit depth {counts['two_qubit_depth']}, total depth {counts['total_depth']}")
    print(f"   transpiled circuit, simulated at max_bond={training_max_bond_32q}, vs target (max_bond={target_max_bond_32q}): "
          f"fidelity = {counts['transpiled_fidelity']:.6f}  (qubit order unchanged: {counts['qubit_order_unchanged']})")

    # recheck: simulate the same trained circuit once more with more room, and compare with the same target
    for recheck_bond in [256, 512]:
        trained_mps = tensornetwork_from_circuit(aqc_ansatz_32q[label].assign_parameters(trained_angles_32q[label]), make_simulator_settings(max_bond=recheck_bond))
        print(f"   recheck: trained circuit simulated at max_bond={recheck_bond} (training used {training_max_bond_32q}), "
              f"vs target (max_bond={target_max_bond_32q}): fidelity = {mps_fidelity(trained_mps, targets_32q[label]):.6f}")



32 qubits, vacuum, original 10-step target circuit: 1796 CNOTs, two-qubit depth 147, total depth 347  (transpiled to a linear chain, optimisation level 3)
32 qubits, vacuum, trained AQC circuit (3-step start): 611 CNOTs, two-qubit depth 48, total depth 175
   transpiled circuit, simulated at max_bond=128, vs target (max_bond=256): fidelity = 0.999008  (qubit order unchanged: True)
   recheck: trained circuit simulated at max_bond=256 (training used 128), vs target (max_bond=256): fidelity = 0.999216
   recheck: trained circuit simulated at max_bond=512 (training used 128), vs target (max_bond=256): fidelity = 0.999217

32 qubits, meson, original 10-step target circuit: 1796 CNOTs, two-qubit depth 147, total depth 347  (transpiled to a linear chain, optimisation level 3)
32 qubits, meson, trained AQC circuit (3-step start): 611 CNOTs, two-qubit depth 48, total depth 174
   transpiled circuit, simulated at max_bond=128, vs target (max_bond=256): fidelity = 0.999499  (qubit order unchang

## Summary (32 qubits)
<small>The trained circuit started as the same Trotter circuit with only 3 steps instead of 10 (each step 10/3 times bigger), and was trained to match the 10-step target. At 32 qubits it still matches the target to 99.9%, with about 3× fewer CNOTs.</small>

| | vacuum | meson |
|---|---|---|
| target `max_bond` (no change from 256 to 512) | 256 | 256 |
| fidelity, training at `max_bond` 128 | 0.99901 | 0.99950 |
| fidelity, same trained circuit checked at `max_bond` 256 | 0.99922 | 0.99951 |
| fidelity, same trained circuit checked at `max_bond` 512 | 0.99922 | 0.99951 |
| CNOTs: 10-step → trained | 1796 → 611 | 1796 → 611 |
| two-qubit depth | 147 → 48 | 147 → 48 |
| total depth | 347 → 175 | 347 → 174 |
| training time | 4.1 min | 3.8 min |

# 80 qubits (40 sites, 10 Trotter steps, meson only)
The largest size reached by the parallel implementation, for a like-for-like comparison. *(Planned, optional.)*

1. Build the target
2. Check convergence
3. Build the cheap trainable circuit
4. Train the AQC angles to match the target
5. Count CNOTs and depth

# 120 qubits (60 sites, 20 Trotter steps)
The benchmark size. Only targets for now; training needs a cluster.

## Settings (120 qubits)
Everything this section changes, in one place.

In [12]:
# Settings (120 qubits)
big_n_sites = 60
big_num_qubits = 2 * big_n_sites
big_num_trotter_steps = 20
big_max_bonds = [256, 512, 1024]

## Step 1 — Build the targets (120 qubits)
Vacuum and meson at `max_bond` 256 / 512 / 1024, each saved to `targets_120qubits_merged/`. About 6.5 hours in total.

In [13]:
# Step 1 (120 qubits) — build the targets (60 sites, 20 steps)
if not build_120_qubit_targets:
    print("120-qubit Step 1 skipped: set build_120_qubit_targets = True in the Switches cell to rebuild the 120-qubit targets (about 6.5 hours).")
else:
    import os, time, pickle

    os.makedirs("targets_120qubits_merged", exist_ok=True)

    for use_meson_not_vacuum in [False, True]:
        label = "meson" if use_meson_not_vacuum else "vacuum"
        previous_chances_filled_per_seat = None

        for max_bond in big_max_bonds:
            file_name = f"targets_120qubits_merged/target_{label}_maxbond{max_bond}.pkl"
            start_time = time.time()
        
            big_target_mps = build_target_using_lib(n_sites=big_n_sites, num_trotter_steps=big_num_trotter_steps, max_bond=max_bond, use_meson_not_vacuum=use_meson_not_vacuum)
        
            norm_violation, bond_chances_filled_per_seat, count_of_filled_seats = check_target(big_target_mps)

            with open(file_name, "wb") as f:
                pickle.dump({"psi": big_target_mps.psi, "chances_filled_per_seat": bond_chances_filled_per_seat, "norm_violation": norm_violation, "count_of_filled_seats": count_of_filled_seats}, f)

            change_vs_previous_chances = "  -   " if previous_chances_filled_per_seat is None else f"{max(abs(np.array(bond_chances_filled_per_seat) - np.array(previous_chances_filled_per_seat))):.4f}"

            print(f"{label:6s} max_bond={max_bond:3d}: {time.time() - start_time:5.0f} s  norm violation={norm_violation:.4f}  "
                  f"count of filled seats={count_of_filled_seats:.4f} (must be {big_n_sites})  biggest seat change vs previous cap={change_vs_previous_chances}")
            previous_chances_filled_per_seat = bond_chances_filled_per_seat
        


120-qubit Step 1 skipped: set build_120_qubit_targets = True in the Switches cell to rebuild the 120-qubit targets (about 6.5 hours).


## Summary (120 qubits)
At `max_bond` 1024: norm left 0.985 (vacuum) / 0.988 (meson); the chance of each qubit reading 1 changes by at most 0.0054 / 0.0056 from 512 to 1024. Training not done yet (needs a cluster).